In [ ]:
"""
SUPERSEDED on 2026-10-06 by notebooks/step03_raw_data_check.ipynb and scripts/find_ohlcv_data_issues.py.

Kept unchanged (code and saved outputs) as the record of how the raw data was downloaded before that date. It does not
run any more: it imports the old root modules (datetime_utils, local_file_management) that were replaced by the so
package of stock_overflow_workspace. The bugs it had are listed in README.md ("What changed").
"""
None

In [1]:
import pandas as pd
import numpy as np
import tzlocal
import pandas_market_calendars as mcal
import pytz
# IMPORT MARKET DATETIME FUNCTIONS
from datetime_utils import get_date_range_market_schedule_pdf,\
                            get_date_market_schedule_pdf
# IMPORT LOCAL FILE MANAGEMENT
from local_file_management import get_path_file_list,\
                                    read_pdf_from_csv_file_path_list

In [2]:
# GET THE LOCAL TIMEZONE NAME
local_tz_name = tzlocal.get_localzone_name()
# COLLECT LOCAL TIMEZONE OBJECT
local_tz = pytz.timezone(local_tz_name)
# GET THE NEW YORK TIMEZONE
ny_tz = pytz.timezone("America/New_York")
# COLLECT THE NYSE CALENDAR
nyse_calendar = mcal.get_calendar("NYSE")

In [5]:
# DEFINE SAVE DATA FILE PATH
OHLCV_DATA_FILE_PATH_STR = "//100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data/"

In [19]:
# CALL FUNCTION TO GET ALL FILES IN PATH
ohlcv_data_file_path_list = get_path_file_list(OHLCV_DATA_FILE_PATH_STR)
# PREVIEW LIST
ohlcv_data_file_path_list

['//100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data/ohlcv_data_2009.csv',
 '//100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data/ohlcv_data_2010.csv',
 '//100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data/ohlcv_data_2011.csv',
 '//100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data/ohlcv_data_2012.csv',
 '//100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data/ohlcv_data_2013.csv',
 '//100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data/ohlcv_data_2014.csv',
 '//100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data/ohlcv_data_2015.csv',
 '//100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data/ohlcv_data_2016.csv',
 '//100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data/ohlcv_data_2017.csv',
 '//100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data/ohlcv_data_2018.csv',
 '//100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data/ohlcv_data

In [20]:
# CALL FUNCTION TO READ PDF FROM PATH LIST
raw_ohlcv_pdf = read_pdf_from_csv_file_path_list(ohlcv_data_file_path_list)
# CONVERT THE TIMESTAMP COLUMN TO TIMESTAMP
raw_ohlcv_pdf["timestamp"] = pd.to_datetime(raw_ohlcv_pdf["timestamp"], utc=True).dt.tz_convert(ny_tz)
raw_ohlcv_pdf["date"] = pd.to_datetime(raw_ohlcv_pdf["date"])
# PREVIEW DATAFRAME
raw_ohlcv_pdf

,timestamp,open,high,low,close,volume,created_ts,date
0,2009-02-23 09:30:00-05:00,78.25,78.27,78.12,78.13,1369700,NaN,2009-02-23
1,2009-02-23 09:31:00-05:00,78.13,78.20,78.09,78.17,861600,NaN,2009-02-23
2,2009-02-23 09:32:00-05:00,78.18,78.22,78.10,78.14,1013200,NaN,2009-02-23
3,2009-02-23 09:33:00-05:00,78.14,78.15,78.05,78.08,1107200,NaN,2009-02-23
4,2009-02-23 09:34:00-05:00,78.07,78.09,77.93,77.95,907000,NaN,2009-02-23
...,...,...,...,...,...,...,...,...
1707032,2026-08-11 15:55:00-04:00,770.41,770.48,770.15,770.24,175472,NaN,2026-08-11
1707033,2026-08-11 15:56:00-04:00,770.23,770.34,770.06,770.15,138197,NaN,2026-08-11
1707034,2026-08-11 15:57:00-04:00,770.17,770.20,770.05,770.08,136883,NaN,2026-08-11
1707035,2026-08-11 15:58:00-04:00,770.08,770.33,769.97,770.33,297192,NaN,2026-08-11


In [16]:
# AGGREGATE THE DATAFRAME BY DATE
data_market_schedule_pdf = raw_ohlcv_pdf.groupby("date").agg(data_market_open_ts=("timestamp", "min"), data_market_close_ts=("timestamp", "max")).reset_index()
# CALCULATE THE PERIOD COUNT
data_market_schedule_pdf["data_period_count"] = ((data_market_schedule_pdf["data_market_close_ts"] - data_market_schedule_pdf["data_market_open_ts"]).dt.total_seconds() / 60) + 1
# PREVIEW DATAFRAME
data_market_schedule_pdf

,date,data_market_open_ts,data_market_close_ts,data_period_count
0,2009-02-23,2009-02-23 09:30:00-05:00,2009-02-23 15:59:00-05:00,390.0
1,2009-02-24,2009-02-24 09:30:00-05:00,2009-02-24 15:59:00-05:00,390.0
2,2009-02-25,2009-02-25 09:30:00-05:00,2009-02-25 15:59:00-05:00,390.0
3,2009-02-26,2009-02-26 09:30:00-05:00,2009-02-26 15:59:00-05:00,390.0
4,2009-02-27,2009-02-27 09:30:00-05:00,2009-02-27 15:59:00-05:00,390.0
...,...,...,...,...
4389,2026-08-05,2026-08-05 09:30:00-04:00,2026-08-05 15:59:00-04:00,390.0
4390,2026-08-06,2026-08-06 09:30:00-04:00,2026-08-06 15:59:00-04:00,390.0
4391,2026-08-07,2026-08-07 09:30:00-04:00,2026-08-07 15:59:00-04:00,390.0
4392,2026-08-10,2026-08-10 09:30:00-04:00,2026-08-10 15:59:00-04:00,390.0


In [13]:
# COLLECT MARKET START AND END DATE
market_start_date = raw_ohlcv_pdf["date"].min()
market_end_date = raw_ohlcv_pdf["date"].max()
# CALL FUNCTION TO COLLECT MARKET SCHEDULE DATAFRAME
expected_market_schedule_pdf = get_date_range_market_schedule_pdf(market_start_date, market_end_date)
# CALCULATE THE PERIOD COUNT
expected_market_schedule_pdf["period_count"] = ((expected_market_schedule_pdf["market_close_ts"] - expected_market_schedule_pdf["market_open_ts"]).dt.total_seconds() / 60) + 1
# PREVIEW DATAFRAME
expected_market_schedule_pdf

,date,market_open_ts,market_close_ts,period_count
0,2009-02-23,2009-02-23 09:30:00-05:00,2009-02-23 15:59:00-05:00,390.0
1,2009-02-24,2009-02-24 09:30:00-05:00,2009-02-24 15:59:00-05:00,390.0
2,2009-02-25,2009-02-25 09:30:00-05:00,2009-02-25 15:59:00-05:00,390.0
3,2009-02-26,2009-02-26 09:30:00-05:00,2009-02-26 15:59:00-05:00,390.0
4,2009-02-27,2009-02-27 09:30:00-05:00,2009-02-27 15:59:00-05:00,390.0
...,...,...,...,...
4389,2026-08-05,2026-08-05 09:30:00-04:00,2026-08-05 15:59:00-04:00,390.0
4390,2026-08-06,2026-08-06 09:30:00-04:00,2026-08-06 15:59:00-04:00,390.0
4391,2026-08-07,2026-08-07 09:30:00-04:00,2026-08-07 15:59:00-04:00,390.0
4392,2026-08-10,2026-08-10 09:30:00-04:00,2026-08-10 15:59:00-04:00,390.0


In [17]:
# MERGE THE EXPECTED AND DATA MARKET SCHEDULE DATAFRAMES
compare_market_schedule_pdf = expected_market_schedule_pdf.merge(data_market_schedule_pdf, on="date", how="left")
# PREVIEW DATAFRAME
compare_market_schedule_pdf

,date,market_open_ts,market_close_ts,period_count,data_market_open_ts,data_market_close_ts,data_period_count
0,2009-02-23,2009-02-23 09:30:00-05:00,2009-02-23 15:59:00-05:00,390.0,2009-02-23 09:30:00-05:00,2009-02-23 15:59:00-05:00,390.0
1,2009-02-24,2009-02-24 09:30:00-05:00,2009-02-24 15:59:00-05:00,390.0,2009-02-24 09:30:00-05:00,2009-02-24 15:59:00-05:00,390.0
2,2009-02-25,2009-02-25 09:30:00-05:00,2009-02-25 15:59:00-05:00,390.0,2009-02-25 09:30:00-05:00,2009-02-25 15:59:00-05:00,390.0
3,2009-02-26,2009-02-26 09:30:00-05:00,2009-02-26 15:59:00-05:00,390.0,2009-02-26 09:30:00-05:00,2009-02-26 15:59:00-05:00,390.0
4,2009-02-27,2009-02-27 09:30:00-05:00,2009-02-27 15:59:00-05:00,390.0,2009-02-27 09:30:00-05:00,2009-02-27 15:59:00-05:00,390.0
...,...,...,...,...,...,...,...
4389,2026-08-05,2026-08-05 09:30:00-04:00,2026-08-05 15:59:00-04:00,390.0,2026-08-05 09:30:00-04:00,2026-08-05 15:59:00-04:00,390.0
4390,2026-08-06,2026-08-06 09:30:00-04:00,2026-08-06 15:59:00-04:00,390.0,2026-08-06 09:30:00-04:00,2026-08-06 15:59:00-04:00,390.0
4391,2026-08-07,2026-08-07 09:30:00-04:00,2026-08-07 15:59:00-04:00,390.0,2026-08-07 09:30:00-04:00,2026-08-07 15:59:00-04:00,390.0
4392,2026-08-10,2026-08-10 09:30:00-04:00,2026-08-10 15:59:00-04:00,390.0,2026-08-10 09:30:00-04:00,2026-08-10 15:59:00-04:00,390.0


In [22]:
# COLLECT ALL DAYS WHERE THE PERIOD COUNT IS NOT EQUAL TO THE DATA PERIOD COUNT
problem_data_pdf = compare_market_schedule_pdf[compare_market_schedule_pdf["period_count"] != compare_market_schedule_pdf["data_period_count"]]
# PREVIEW DATAFRAME
problem_data_pdf

,date,market_open_ts,market_close_ts,period_count,data_market_open_ts,data_market_close_ts,data_period_count
107,2009-07-27,2009-07-27 09:30:00-04:00,2009-07-27 15:59:00-04:00,390.0,2009-07-27 11:15:00-04:00,2009-07-27 15:59:00-04:00,285.0
1218,2013-12-23,2013-12-23 09:30:00-05:00,2013-12-23 15:59:00-05:00,390.0,2013-12-23 10:08:00-05:00,2013-12-23 15:59:00-05:00,352.0
